In [3]:
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")

print("pandas version:", pd.__version__)

# Load the teen mental health CSV whether the kernel starts in Week5/ or the repo root.
_csv_candidates = [
    Path("Teen_Mental_Health_Dataset.csv"),
    Path("Week5/Teen_Mental_Health_Dataset.csv"),
]
_csv_path = next((p for p in _csv_candidates if p.exists()), None)
if _csv_path is None:
    raise FileNotFoundError(
        "Could not find Teen_Mental_Health_Dataset.csv next to this notebook or under Week5/."
    )
df = pd.read_csv(_csv_path)
print("Loaded rows:", len(df))

pandas version: 1.1.5
Loaded rows: 1200


In [ ]:
# Step 1 — What does this dataset look like, and is it the shape I expect?
# What I'm asking: what are example rows, column names, and dtypes? That tells me each row is one teen,
# which fields are numeric (hours, scales) vs categorical (gender, platform), and whether row count
# matches what I downloaded—so I know later filters and means use the right units.

print("=== df.head() — first rows ===")
print(df.head())

print("\n=== df.info() — columns, dtypes, non-null counts ===")
df.info()

=== df.head() — first rows ===
   age  gender  daily_social_media_hours platform_usage  sleep_hours  \
0   14    male                       7.9      Instagram          7.4   
1   19  female                       1.9         TikTok          8.0   
2   17  female                       1.3      Instagram          7.6   
3   15    male                       7.4         TikTok          6.9   
4   15  female                       4.7           Both          4.9   

   screen_time_before_sleep  academic_performance  physical_activity  \
0                       2.9                  3.01                1.5   
1                       2.9                  3.22                0.8   
2                       0.5                  3.92                0.0   
3                       1.6                  3.48                0.8   
4                       3.0                  2.37                1.4   

  social_interaction_level  stress_level  anxiety_level  addiction_level  \
0                      low 

In [ ]:
# Step 2 — Are gender and main platform balanced, or skewed?
# What I'm asking: what are the most common values? That tells me if one gender or one app dominates;
# if counts are very uneven, averages by group later are driven mostly by the majority bucket.

print("=== value_counts: gender ===")
print(df["gender"].value_counts())

print("\n=== value_counts: platform_usage ===")
print(df["platform_usage"].value_counts())

=== value_counts: gender ===
male      615
female    585
Name: gender, dtype: int64

=== value_counts: platform_usage ===
Instagram    411
TikTok       398
Both         391
Name: platform_usage, dtype: int64


In [ ]:
# Step 3 — How complete is the data?
# What I'm asking: how many missing cells per column? Large gaps in sleep or stress would weaken
# summaries or force drops; zeros mean those columns are complete enough for simple stats.

print("=== isnull().sum() — missing values per column ===")
print(df.isnull().sum())

=== isnull().sum() — missing values per column ===
age                         0
gender                      0
daily_social_media_hours    0
platform_usage              0
sleep_hours                 0
screen_time_before_sleep    0
academic_performance        0
physical_activity           0
social_interaction_level    0
stress_level                0
anxiety_level               0
addiction_level             0
depression_label            0
dtype: int64


In [ ]:
# Step 4 — What does a high-stress subset look like?
# What I'm asking: who has stress_level above 6? The count and sample rows show how many teens are in
# that band and whether sleep or depression_label look different at a glance vs the full table.

print("=== filter: stress_level > 6 ===")
high_stress = df["stress_level"] > 6
print("Number of teens with stress_level > 6:", int(high_stress.sum()))
print(df.loc[high_stress, ["age", "gender", "stress_level", "sleep_hours", "depression_label"]].head(10))

=== filter: stress_level > 6 ===
Number of teens with stress_level > 6: 460
    age  gender  stress_level  sleep_hours  depression_label
1    19  female             8          8.0                 0
17   16    male            10          6.1                 0
18   16  female            10          6.8                 0
19   14  female             8          7.6                 0
21   18    male             8          7.4                 0
23   18  female             9          7.6                 0
24   18  female             8          6.7                 0
26   14  female             8          8.7                 0
27   19    male             7          7.1                 0
31   13  female             8          8.7                 0


In [ ]:
# Step 5 — Do average sleep hours differ by gender in this sample?
# What I'm asking: mean sleep_hours by gender. That shows a simple central-tendency gap between groups,
# not causation—only whether a difference exists worth plotting or discussing further.

print("=== groupby: mean sleep_hours by gender ===")
print(df.groupby("gender")["sleep_hours"].mean())

=== groupby: mean sleep_hours by gender ===
gender
female    6.496410
male      6.404715
Name: sleep_hours, dtype: float64


In [ ]:
# Step 6 — Does mean stress differ by main platform?
# What I'm asking: average stress_level for each platform_usage value. If one platform lines up with
# higher mean stress, that is a descriptive pattern to explore (many confounds in real life).

print("=== groupby: mean stress_level by platform_usage ===")
print(df.groupby("platform_usage")["stress_level"].mean())

=== groupby: mean stress_level by platform_usage ===
platform_usage
Both         5.549872
Instagram    5.498783
TikTok       5.288945
Name: stress_level, dtype: float64
